In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import openai
import os

os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)
os.environ.pop("ALL_PROXY", None)

os.environ["NO_PROXY"] = "*"

if not os.path.exists("test_notebooks"):
    os.chdir("..")

assert os.path.exists("test_notebooks")

In [ ]:

client = openai.OpenAI(
    # This is the default and can be omitted
    base_url = "http://localhost:8080/api",
    api_key= "sk-9f605d210924406d8e60b10122f59e96", # open-webui, safe
)

In [ ]:
autodl_kimi = "Kimi-K2.5" # 这个是autodl的

# tion.choices[0].message.content)

In [ ]:
# demo

import base64
import os
from pathlib import Path
import io
from PIL import Image


def encode_imgfile_to_url(file: str, quality=90, debug=False) -> str:
    with io.BytesIO() as buf:
        with Image.open(file) as im:
            im.convert("RGB").save(buf, format="JPEG", quality=quality)

            if debug:
                Path("jpg-debug.tmp.jpg").write_bytes(buf.getvalue())

            image_url = f"data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode('utf-8')}"

            return image_url

a = encode_imgfile_to_url("runs/episodes/miyako_1/images/t008_02.png", debug=True)

a.__len__() // 1024, "KB-b64"

In [ ]:
if False:
    completion = client.chat.completions.create(
        model=autodl_kimi,
        temperature=0.6,
        extra_body={
            "thinking": {"type": "disabled"},
        },
        messages=[
            {
                "role": "user",
                # 注意这里，content 由原来的 str 类型变更为一个 list，这个 list 中包含多个部分的内容，图片（image_url）是一个部分（part），
                # 文字（text）是一个部分（part）
                "content": [
                    {
                        "type": "text",
                        "text": "img_g1:",
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": encode_imgfile_to_url(r"runs\episodes\char_sora\t003_01.png"),
                        },
                    },
                    {
                        "type": "text",
                        "text": "img_x1:",
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": encode_imgfile_to_url("runs/episodes/miyako_1/images/t008_02.png"),
                        },
                    },
                    {
                        "type": "text",
                        "text": "你看了哪几个img？名称叫什么？简单描述一下区别？"
                    }
                ],
            },
        ],
    )

    print("completion:", completion)
    
    print(completion.choices[0].message.content)
    print("usage", completion.usage)

artist only phase comparison

In [ ]:
from typing import List


test_case = {
    "ref_image": r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t004_02.png",
    "ref_text": "我喜欢这种美少女图片",
    "candidates": [
        r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t001_03.png",  # 还行 [0]
        r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t002_00.png",  # 不好 [1]
        r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t002_01.png",  # 不好 [2]
        r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t002_03.png",  # 还行 [3]
    ],
}


def llm_compare_image_artist_only(ref_image: str, ref_text: str, candidates: List[str], temperature=0.6) -> str:
    system_message = """
## 角色
你是一个二次元图像审美评判专家。你需要为用户选择更好的图片。用户会提交用户需求。
用户需求由 参考图片ref_image、文本需求组成。

除此之外，你还有 待评价图片 candidate[i] 作为输入。

你要做的是，根据用户的参考图片、文本需求，将待评价图片进行排序，由好到差。

## 输出
1. 分析并说明理由。分析的时候，重点注意：ref_image不参与排序，以及candidate的数量，切勿弄混。
2. 得出结论：尝试推测用户喜好。待评价图片从好到差。你需要寻找用户可能最喜欢的图片。以代码块compare的格式输出。

```compare
candidate[?] > candidate[?] > ...
```

"""
    user_image_contents = [
        {
            "type": "text",
            "text": f"用户需求文本: {ref_text}, 用户提供的参考图片 ref_image (不参与评分排序):",
        },
        {
            "type": "image_url",
            "image_url": {
                "url": encode_imgfile_to_url(ref_image),
            },
        },
        {
            "type": "text",
            "text": f"\n------\n以下是所有待评价图片(candidate[0] - candidate[{len(candidates)-1}]):"
        }
    ]

    for i, candidate in enumerate(candidates):
        user_image_contents += [
            {
                "type": "text",
                "text": f"candidate[{i}]:",
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": encode_imgfile_to_url(candidate),
                },
            },
        ]
        

    completion = client.chat.completions.create(
        model=autodl_kimi,
        temperature=temperature,
        extra_body={
            "thinking": {"type": "disabled"},
        },
        messages=[
            {
                "role": "system",  # type: ignore
                "content": system_message,
            },
            {
                "role": "user",
                "content": user_image_contents,
            },
        ],
    )

    print("usage:", completion.usage)

    s = completion.choices[0].message.content
    assert s
    return s

# out = llm_compare_image_artist_only(test_case["ref_image"], test_case["ref_text"], test_case["candidates"] )
# print(out)

del test_case

compare within a timestep

In [ ]:
# modify this every time!
prototyping_params: dict = {
    "episode_name": "kimi_director_3",
    "ref_image": r"C:\Users\ThePlayer\Desktop\compare\large-target.jpg", 
    "ref_text": "我喜欢这种风格的美少女，给我多来几张。",
}

In [ ]:
from entropy.infra.episode_repository import EpisodeRepository
import re

timestep_case = dict(prototyping_params)

timestep_case.update({
    "timestep": -1,
    "temperature": 0.0, 
})

# get all images of timestep
timesteps_q = EpisodeRepository.get_timesteps_query_model(timestep_case["episode_name"])
timesteps_q[timestep_case["timestep"]]

candidates = [p.local_abs_path for p in timesteps_q[timestep_case["timestep"]].images]
# candidates

out = llm_compare_image_artist_only(timestep_case["ref_image"], timestep_case["ref_text"], candidates, timestep_case["temperature"])
print(out)


# parse output
# print(out)

pattern = r"```compare\s+(.*?)\s+```"
match = re.search(pattern, out, re.DOTALL)
assert match, 'no match'

compare_result = match.group(1).strip()

# integrity
for i in range(len(candidates)):
    ss = f'candidate[{i}]'
    assert ss in compare_result, f'{ss} not found'

assert compare_result.count("candidate") == len(candidates), 'candidate count error'


print("\n\ncompare_result:", compare_result)
print(f"\nNOTE: episode: {timestep_case['episode_name']}, timestep: {timestep_case['timestep']}")

del candidates, timesteps_q, timestep_case


free eval

In [ ]:
def llm_compare_image_free(ref_image: str, ref_text: str, candidates: List[str], temperature=0.6) -> str:
    """
    我感觉这个temperature没那么重要了，0.6就行
    """
    
    system_message = """
## 角色
你是一个二次元图像审美评判专家。你需要为用户选择更好的图片。用户会提交用户需求。
用户需求由 参考图片ref_image、文本需求组成。

除此之外，你还有 待评价图片 candidate[i] 作为输入。

你要做的是，根据用户的参考图片、文本需求，将待评价图片进行排序，由好到差。

## 输出
1. 简短搞清楚candidate的数量 不要将ref_image算在里面
2. 分析并说明理由。对比维度：不止画风，还有 自然环境、外貌、动作、心理、神态、整体外貌、容貌五官、衣着服饰、姿态神情。要充分考虑用户喜欢什么

3. 进行系统化的评价（用代码块包裹）不能提到参考图，假装没有参考图，让阅读者认为，一切都是出于你的个人喜好。
    - [相对评价] 假装你本身就是用户，给出你这样评价的理由。多用“我”。
        - 不能提到参考图，也不要描述参考图的特点，不要直接透露你的喜好（我喜欢xx样子的图片），而是要说哪张图哪里好/不好，间接透露。
        - 格式需简单，不要分点换行等。150字以内。
    - [绝对误差评价] 评价完成后，给出一个你认为待评价图片总体来说都最欠缺的点，或者完全没有考虑到的点（加上这个，用户可能会更喜欢），同样用第一人称。限制100字。以建议的语气。完了之后给3-5个关键词，用于表达中心思想。
```commentary
这里以文本的形式，进行相对评价、绝对误差评价。使用candidate[i]指代图片避免歧义。
```
    
3. 输出排名在compare代码块中
```rank
candidate[?] > candidate[?] > ...
```

"""
    user_image_contents = [
        {
            "type": "text",
            "text": f"用户需求文本: {ref_text}, 用户提供的参考图片 ref_image (不参与评分排序):",
        },
        {
            "type": "image_url",
            "image_url": {
                "url": encode_imgfile_to_url(ref_image),
            },
        },
        {
            "type": "text",
            "text": f"\n------\n以下是所有待评价图片(candidate[0] - candidate[{len(candidates)-1}]):"
        }
    ]

    for i, candidate in enumerate(candidates):
        user_image_contents += [
            {
                "type": "text",
                "text": f"这个图片是 candidate[{i}]:",
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": encode_imgfile_to_url(candidate),
                },
            },
        ]

    completion = client.chat.completions.create(
        model=autodl_kimi,
        temperature=temperature,
        extra_body={
            "thinking": {"type": "disabled"},
        },
        messages=[
            {
                "role": "system",  # type: ignore
                "content": system_message,
            },
            {
                "role": "user",
                "content": user_image_contents,
            },
        ],
    )

    print("usage:", completion.usage)

    s = completion.choices[0].message.content
    assert s
    return s


# test_case = {
#     "ref_image": r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t004_02.png",
#     "ref_text": "我喜欢这种美少女图片",
#     "candidates": [
#         r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t001_03.png",  # 还行 [0]
#         r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t002_00.png",  # 不好 [1]
#         r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t002_01.png",  # 不好 [2]
#         r"C:\Users\ThePlayer\Desktop\ImageAutoPrompt\runs\episodes\kimi_modal_1\images\t002_03.png",  # 还行 [3]
#     ],
# }

# out = llm_compare_image_free(test_case["ref_image"], test_case["ref_text"], test_case["candidates"] )
# print(out)

# del test_case

In [ ]:
from entropy.infra.episode_repository import EpisodeRepository
import re

timestep_case = dict(prototyping_params)

timestep_case.update({
    "timestep": -1,
    "temperature": 0.6,
})

# ------------------------------------------

# get all images of timestep
timesteps_q = EpisodeRepository.get_timesteps_query_model(timestep_case["episode_name"])
timesteps_q[timestep_case["timestep"]]

candidates = [p.local_abs_path for p in timesteps_q[timestep_case["timestep"]].images]
# candidates

out = llm_compare_image_free(timestep_case["ref_image"], timestep_case["ref_text"], candidates, timestep_case["temperature"])
print(out)

del timestep_case

# validation --------------------

# rank block
pattern = r"```rank\s+(.*?)\s+```"
match = re.search(pattern, out, re.DOTALL)
assert match, 'rank no match'

compare_result = match.group(1).strip()
# print("\n\ncompare_result:", compare_result)

# integrity
for i in range(len(candidates)):
    ss = f'candidate[{i}]'
    assert ss in compare_result, f'{ss} not found'

assert compare_result.count("candidate") == len(candidates), 'candidate count error'

# commentary
pattern = r"```commentary\s+(.*?)\s+```"
match = re.search(pattern, out, re.DOTALL)
assert match, 'commentary no match'
commentary = match.group(1).strip()

del candidates, timesteps_q


In [ ]:
print(f"{commentary}\n{compare_result}")